# Rebalance Frequency and Transaction Cost Test

This notebook checks what happens when factor portfolios are rebalanced every `n` months, where `n = 1, 2, ..., 12`.

Interpretation used here:

- `n = 1` is the current monthly rebalance baseline.
- `n = 3` means rebalance only once every 3 months.
- Between rebalance months, the old holdings are carried forward and allowed to drift with stock returns.
- Transaction cost is charged only when the portfolio is actually rebalanced.
- The transaction-cost drag follows the website logic: `turnover * bps / 10000`.
- The first formation month is not charged a transaction cost, matching the current website default.

The notebook runs the test for individual factor portfolios, across universes and transaction-cost examples, then exports CSV/XLSX summaries and a few simple charts.


In [ ]:
from pathlib import Path
import json
import os

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "Data" / "Derived" / "backtest-runtime.json").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "Data" / "Derived" / "backtest-runtime.json").exists():
    raise FileNotFoundError("Run this notebook from inside factorboosting.github.io.")

os.environ.setdefault("XDG_CACHE_HOME", str(REPO_ROOT / ".cache"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib-cache"))
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUT_DIR = REPO_ROOT / "notebooks" / "rebalance_frequency_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

START_MONTH = "2003-10"
END_MONTH = "2026-05"
UNIVERSES = ["all", "top500", "top300"]
REBALANCE_MONTHS = list(range(1, 13))
TRANSACTION_COST_BPS = [0, 10, 20, 50, 100]
WEIGHTINGS = ["vw"]  # change to ["ew", "vw"] if you want both
MIN_FIRMS_PER_SIZE_BUCKET = 5

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.facecolor": "white", "axes.grid": True, "grid.color": "#eef2f7"})

print(f"Repo root: {REPO_ROOT}")
print(f"Output dir: {OUT_DIR}")


In [ ]:
# Individual factor portfolios. Each one is intentionally simple and editable.
# long_short rows are classic factor spreads; long_only rows are the corresponding good-leg portfolios.
PORTFOLIOS = [
    {"name": "SMB small-minus-big", "strategy": "long_short", "long": {"Size": ["S"]}, "short": {"Size": ["B"]}},
    {"name": "HML value-minus-growth", "strategy": "long_short", "long": {"Book-to-Market": ["V"]}, "short": {"Book-to-Market": ["G"]}},
    {"name": "RMW robust-minus-weak", "strategy": "long_short", "long": {"Profitability": ["R"]}, "short": {"Profitability": ["W"]}},
    {"name": "CMA conservative-minus-aggressive", "strategy": "long_short", "long": {"Investment": ["C"]}, "short": {"Investment": ["A"]}},
    {"name": "MOM winner-minus-loser", "strategy": "long_short", "long": {"Momentum": ["W"]}, "short": {"Momentum": ["L"]}},
    {"name": "AT high-minus-low", "strategy": "long_short", "long": {"Asset Turnover": ["H"]}, "short": {"Asset Turnover": ["L"]}},
    {"name": "SG high-minus-low", "strategy": "long_short", "long": {"Sales Growth": ["H"]}, "short": {"Sales Growth": ["L"]}},
    {"name": "ACC conservative-minus-aggressive", "strategy": "long_short", "long": {"Accruals": ["C"]}, "short": {"Accruals": ["A"]}},
    {"name": "VOL low-minus-high", "strategy": "long_short", "long": {"Volatility": ["L"]}, "short": {"Volatility": ["H"]}},
    {"name": "STR loser-minus-winner", "strategy": "long_short", "long": {"Short-Term Reversal": ["L"]}, "short": {"Short-Term Reversal": ["H"]}},

    {"name": "Small", "strategy": "long_only", "long": {"Size": ["S"]}, "short": {}},
    {"name": "Value", "strategy": "long_only", "long": {"Book-to-Market": ["V"]}, "short": {}},
    {"name": "Robust profitability", "strategy": "long_only", "long": {"Profitability": ["R"]}, "short": {}},
    {"name": "Conservative investment", "strategy": "long_only", "long": {"Investment": ["C"]}, "short": {}},
    {"name": "Momentum winner", "strategy": "long_only", "long": {"Momentum": ["W"]}, "short": {}},
    {"name": "High asset turnover", "strategy": "long_only", "long": {"Asset Turnover": ["H"]}, "short": {}},
    {"name": "High sales growth", "strategy": "long_only", "long": {"Sales Growth": ["H"]}, "short": {}},
    {"name": "Conservative accruals", "strategy": "long_only", "long": {"Accruals": ["C"]}, "short": {}},
    {"name": "Low volatility", "strategy": "long_only", "long": {"Volatility": ["L"]}, "short": {}},
    {"name": "Short-term reversal loser", "strategy": "long_only", "long": {"Short-Term Reversal": ["L"]}, "short": {}},
]

FACTOR_LABEL_COL = {
    "Book-to-Market": "BM_Label",
    "Profitability": "OP_Label",
    "Investment": "INV_Label",
    "Momentum": "MOM_Label",
    "Asset Turnover": "AT_Label",
    "Sales Growth": "SG_Label",
    "Accruals": "ACC_Label",
    "Volatility": "VOL_Label",
    "Short-Term Reversal": "STR_Label",
}
PORTFOLIO_CODE = {
    "Profitability": {"col": "RMW_Portfolio", "R": ["SR", "BR"], "N": ["SN", "BN"], "W": ["SW", "BW"]},
    "Investment": {"col": "CMA_Portfolio", "C": ["SC", "BC"], "N": ["SN", "BN"], "A": ["SA", "BA"]},
}

portfolio_preview = pd.DataFrame(PORTFOLIOS)
display(portfolio_preview)


In [ ]:
# Load the same derived universe snapshots used by the local website backtest engine.
runtime = json.loads((REPO_ROOT / "Data" / "Derived" / "backtest-runtime.json").read_text())
rf_by_month = {month: float(value) for month, value in runtime.get("rfData", {}).items() if value is not None}
all_data = {}

for universe in UNIVERSES:
    month_groups = {}
    for chunk in runtime["universes"][universe]["chunks"]:
        if chunk["lastMonth"] < START_MONTH or chunk["firstMonth"] > END_MONTH:
            continue
        snapshot = json.loads((REPO_ROOT / chunk["file"]).read_text())
        columns = snapshot["columns"]
        for month, rows in snapshot["monthGroups"].items():
            if month < START_MONTH or month > END_MONTH:
                continue
            month_groups[month] = [dict(zip(columns, row)) for row in rows]
    months = sorted(month_groups)
    all_data[universe] = {"months": months, "month_groups": month_groups}
    print(f"{universe}: {len(months)} months, {sum(len(v) for v in month_groups.values()):,} rows")


In [ ]:
summary_rows = []
monthly_rows = []
equity_rows = []

for universe in UNIVERSES:
    months = all_data[universe]["months"]
    month_groups = all_data[universe]["month_groups"]

    for portfolio in PORTFOLIOS:
        strategy = portfolio["strategy"]
        long_filters = portfolio["long"]
        short_filters = portfolio["short"]
        active_factors = set(long_filters) | set(short_filters)
        active_non_size = active_factors - {"Size"}

        # Website-style size column choice, kept explicit here for readability.
        if universe == "all" and active_non_size == {"Profitability"}:
            size_col = "RMW_Portfolio"
        elif universe == "all" and active_non_size == {"Investment"}:
            size_col = "CMA_Portfolio"
        elif active_non_size & {"Momentum", "Volatility", "Short-Term Reversal"}:
            size_col = "Size_Label_Monthly"
        elif "Profitability" in active_non_size:
            size_col = "Size_Label_OP"
        elif "Investment" in active_non_size:
            size_col = "Size_Label_INV"
        elif "Asset Turnover" in active_non_size:
            size_col = "Size_Label_AT"
        elif "Sales Growth" in active_non_size:
            size_col = "Size_Label_SG"
        elif "Accruals" in active_non_size:
            size_col = "Size_Label_ACC"
        else:
            size_col = "Size_Label_Yearly"

        for weighting in WEIGHTINGS:
            for rebalance_n in REBALANCE_MONTHS:
                for tc_bps in TRANSACTION_COST_BPS:
                    cost = tc_bps / 10000
                    weights = {}
                    returns = []
                    turnover_list = []
                    long_counts = []
                    short_counts = []
                    equity = 100.0

                    for month_index, month in enumerate(months):
                        rows = month_groups[month]
                        rows_by_code = {str(row["Co_Code"]): row for row in rows}
                        rebalance_now = (month_index == 0) or (month_index % rebalance_n == 0)
                        cost_drag = 0.0

                        if rebalance_now:
                            side_rows = {}
                            for side, filters in [("long", long_filters), ("short", short_filters if strategy == "long_short" else {})]:
                                selected = []
                                for row in rows:
                                    keep = True
                                    for factor, labels in filters.items():
                                        labels = set(labels)
                                        if factor == "Size":
                                            bucket = str(row.get(size_col) or row.get("Size_Label_Yearly") or row.get("Size_Label_Monthly") or row.get("Size_Label") or "")
                                            if size_col in ["RMW_Portfolio", "CMA_Portfolio"]:
                                                bucket = bucket[:1]
                                            if bucket not in labels:
                                                keep = False
                                                break
                                        elif universe == "all" and factor in PORTFOLIO_CODE and active_non_size == {factor} and row.get(PORTFOLIO_CODE[factor]["col"]):
                                            allowed = set()
                                            for label in labels:
                                                allowed.update(PORTFOLIO_CODE[factor][label])
                                            if row.get(PORTFOLIO_CODE[factor]["col"]) not in allowed:
                                                keep = False
                                                break
                                        else:
                                            if row.get(FACTOR_LABEL_COL[factor]) not in labels:
                                                keep = False
                                                break
                                    if keep:
                                        selected.append(row)
                                side_rows[side] = selected

                            long_s = [r for r in side_rows["long"] if str(r.get(size_col) or r.get("Size_Label_Yearly") or r.get("Size_Label_Monthly") or r.get("Size_Label") or "")[:1] == "S"]
                            long_b = [r for r in side_rows["long"] if str(r.get(size_col) or r.get("Size_Label_Yearly") or r.get("Size_Label_Monthly") or r.get("Size_Label") or "")[:1] == "B"]
                            short_s = [r for r in side_rows.get("short", []) if str(r.get(size_col) or r.get("Size_Label_Yearly") or r.get("Size_Label_Monthly") or r.get("Size_Label") or "")[:1] == "S"]
                            short_b = [r for r in side_rows.get("short", []) if str(r.get(size_col) or r.get("Size_Label_Yearly") or r.get("Size_Label_Monthly") or r.get("Size_Label") or "")[:1] == "B"]

                            long_ok_s = len(long_s) >= MIN_FIRMS_PER_SIZE_BUCKET
                            long_ok_b = len(long_b) >= MIN_FIRMS_PER_SIZE_BUCKET
                            short_ok_s = strategy == "long_short" and len(short_s) >= MIN_FIRMS_PER_SIZE_BUCKET
                            short_ok_b = strategy == "long_short" and len(short_b) >= MIN_FIRMS_PER_SIZE_BUCKET
                            if strategy == "long_short" and active_non_size:
                                if not (long_ok_s and short_ok_s):
                                    long_ok_s = short_ok_s = False
                                if not (long_ok_b and short_ok_b):
                                    long_ok_b = short_ok_b = False

                            target = {}
                            for side, bucket_rows, ok in [("long", long_s, long_ok_s), ("long", long_b, long_ok_b), ("short", short_s, short_ok_s), ("short", short_b, short_ok_b)]:
                                if not ok or not bucket_rows:
                                    continue
                                valid_bucket_count = (long_ok_s + long_ok_b) if side == "long" else (short_ok_s + short_ok_b)
                                macro = 1.0 / valid_bucket_count
                                sign = 1.0 if side == "long" else -1.0
                                if weighting == "vw":
                                    bucket_weight = sum(float(r.get("prev_Size") or 0) for r in bucket_rows if float(r.get("prev_Size") or 0) > 0)
                                else:
                                    bucket_weight = 0
                                for row in bucket_rows:
                                    code = str(row["Co_Code"])
                                    if weighting == "vw" and bucket_weight > 0 and float(row.get("prev_Size") or 0) > 0:
                                        w = sign * macro * float(row.get("prev_Size")) / bucket_weight
                                    else:
                                        w = sign * macro / len(bucket_rows)
                                    target[code] = target.get(code, 0.0) + w

                            turnover = sum(abs(target.get(code, 0.0) - weights.get(code, 0.0)) for code in set(target) | set(weights))
                            cost_drag = 0.0 if month_index == 0 else turnover * cost
                            weights = target
                        else:
                            turnover = 0.0

                        gross_return = 0.0
                        for code, weight in weights.items():
                            row = rows_by_code.get(code)
                            gross_return += weight * (float(row.get("_ret") or 0.0) if row else 0.0)
                        net_return = gross_return - cost_drag
                        equity *= 1 + net_return

                        long_count = sum(1 for weight in weights.values() if weight > 0)
                        short_count = sum(1 for weight in weights.values() if weight < 0)
                        returns.append(net_return)
                        turnover_list.append(turnover)
                        long_counts.append(long_count)
                        short_counts.append(short_count)

                        equity_rows.append({
                            "universe": universe,
                            "portfolio": portfolio["name"],
                            "strategy": strategy,
                            "weighting": weighting,
                            "rebalance_months": rebalance_n,
                            "tc_bps": tc_bps,
                            "month": month,
                            "equity": equity,
                        })
                        monthly_rows.append({
                            "universe": universe,
                            "portfolio": portfolio["name"],
                            "strategy": strategy,
                            "weighting": weighting,
                            "rebalance_months": rebalance_n,
                            "tc_bps": tc_bps,
                            "month": month,
                            "return": net_return,
                            "gross_return_before_cost": gross_return,
                            "cost_drag": cost_drag,
                            "turnover": turnover,
                            "long_stocks": long_count,
                            "short_stocks": short_count,
                        })

                        # Let weights drift until the next rebalance. This is the key lagged-rebalance step.
                        drifted = {}
                        denom = 1 + gross_return
                        if denom > 0:
                            for code, weight in weights.items():
                                row = rows_by_code.get(code)
                                stock_ret = float(row.get("_ret") or 0.0) if row else 0.0
                                drifted[code] = weight * (1 + stock_ret) / denom
                        weights = drifted

                    rets = np.array(returns, dtype=float)
                    rfs = np.array([rf_by_month.get(m, 0.0) for m in months], dtype=float)
                    cumulative = float(np.prod(1 + rets))
                    years = len(rets) / 12
                    ann_return = cumulative ** (1 / years) - 1 if years > 0 and cumulative > 0 else np.nan
                    ann_vol = float(np.std(rets, ddof=1) * np.sqrt(12)) if len(rets) > 1 else 0.0
                    excess = rets if strategy == "long_short" else rets - rfs
                    sharpe = float(np.mean(excess) * 12 / ann_vol) if ann_vol > 0 else np.nan
                    curve = np.cumprod(1 + rets)
                    max_dd = float(np.min(curve / np.maximum.accumulate(curve) - 1)) if len(curve) else 0.0

                    summary_rows.append({
                        "universe": universe,
                        "portfolio": portfolio["name"],
                        "strategy": strategy,
                        "weighting": weighting,
                        "rebalance_months": rebalance_n,
                        "tc_bps": tc_bps,
                        "growth_multiple": round(cumulative, 4),
                        "annualized_return": round(ann_return * 100, 2),
                        "annualized_volatility": round(ann_vol * 100, 2),
                        "sharpe_ratio": round(sharpe, 3),
                        "max_drawdown": round(max_dd * 100, 2),
                        "avg_turnover_pct": round(np.mean(turnover_list) * 100, 1),
                        "avg_long_stocks": round(np.mean(long_counts), 1),
                        "avg_short_stocks": round(np.mean(short_counts), 1),
                        "last_long_stocks": long_counts[-1],
                        "last_short_stocks": short_counts[-1],
                        "months": len(months),
                    })

summary = pd.DataFrame(summary_rows)
monthly = pd.DataFrame(monthly_rows)
equity = pd.DataFrame(equity_rows)

baseline = summary[summary["rebalance_months"] == 1][[
    "universe", "portfolio", "strategy", "weighting", "tc_bps",
    "annualized_return", "sharpe_ratio", "max_drawdown", "growth_multiple",
]].rename(columns={
    "annualized_return": "monthly_rebalance_ann_return",
    "sharpe_ratio": "monthly_rebalance_sharpe",
    "max_drawdown": "monthly_rebalance_max_dd",
    "growth_multiple": "monthly_rebalance_growth",
})
summary = summary.merge(baseline, on=["universe", "portfolio", "strategy", "weighting", "tc_bps"], how="left")
summary["ann_return_minus_monthly"] = summary["annualized_return"] - summary["monthly_rebalance_ann_return"]
summary["sharpe_minus_monthly"] = summary["sharpe_ratio"] - summary["monthly_rebalance_sharpe"]
summary["max_dd_minus_monthly"] = summary["max_drawdown"] - summary["monthly_rebalance_max_dd"]
summary["growth_vs_monthly"] = summary["growth_multiple"] / summary["monthly_rebalance_growth"]

summary.sort_values(["universe", "portfolio", "tc_bps", "rebalance_months"]).head(20)


In [ ]:
summary_path = OUT_DIR / "rebalance_frequency_summary.csv"
monthly_path = OUT_DIR / "rebalance_frequency_monthly_returns.csv"
equity_path = OUT_DIR / "rebalance_frequency_equity_curves.csv"
excel_path = OUT_DIR / "rebalance_frequency_results.xlsx"

summary.to_csv(summary_path, index=False)
monthly.to_csv(monthly_path, index=False)
equity.to_csv(equity_path, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    summary.sort_values(["universe", "portfolio", "tc_bps", "rebalance_months"]).to_excel(writer, sheet_name="summary", index=False)
    summary.sort_values(["sharpe_ratio", "annualized_return"], ascending=False).head(200).to_excel(writer, sheet_name="top_200", index=False)
    monthly.to_excel(writer, sheet_name="monthly_returns", index=False)

print(f"Saved {summary_path}")
print(f"Saved {monthly_path}")
print(f"Saved {equity_path}")
print(f"Saved {excel_path}")

display(summary.sort_values(["sharpe_ratio", "annualized_return"], ascending=False).head(25))


In [ ]:
# Simple visual checks. Change these three lines to inspect another universe / cost / factor.
PLOT_UNIVERSE = "all"
PLOT_TC_BPS = 20
PLOT_PORTFOLIOS = ["HML value-minus-growth", "RMW robust-minus-weak", "MOM winner-minus-loser", "Value"]

plot_frame = summary[(summary["universe"] == PLOT_UNIVERSE) & (summary["tc_bps"] == PLOT_TC_BPS)]
heat = plot_frame.pivot_table(index="portfolio", columns="rebalance_months", values="sharpe_ratio", aggfunc="mean")
heat = heat.loc[[p for p in PLOT_PORTFOLIOS if p in heat.index]]

fig, ax = plt.subplots(figsize=(12, max(3.5, 0.45 * len(heat))))
image = ax.imshow(heat.values, aspect="auto", cmap="RdYlGn")
ax.set_xticks(range(len(heat.columns)), heat.columns)
ax.set_yticks(range(len(heat.index)), heat.index)
ax.set_xlabel("Rebalance every n months")
ax.set_title(f"Sharpe by rebalance frequency ({PLOT_UNIVERSE}, {PLOT_TC_BPS} bps)")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        if pd.notna(heat.iloc[i, j]):
            ax.text(j, i, f"{heat.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "rebalance_frequency_sharpe_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

curve_frame = equity[
    (equity["universe"] == PLOT_UNIVERSE)
    & (equity["tc_bps"] == PLOT_TC_BPS)
    & (equity["portfolio"].isin(PLOT_PORTFOLIOS[:3]))
    & (equity["rebalance_months"].isin([1, 3, 6, 12]))
].copy()
curve_frame["date"] = pd.to_datetime(curve_frame["month"] + "-01")

fig, ax = plt.subplots(figsize=(13, 6))
for (portfolio, n), group in curve_frame.groupby(["portfolio", "rebalance_months"]):
    ax.plot(group["date"], group["equity"], linewidth=1.7, label=f"{portfolio}, n={n}")
ax.set_title(f"Equity curves under slower rebalancing ({PLOT_UNIVERSE}, {PLOT_TC_BPS} bps)")
ax.set_ylabel("Portfolio value, start = 100")
ax.legend(fontsize=8, frameon=False, ncol=2)
plt.tight_layout()
plt.savefig(OUT_DIR / "rebalance_frequency_equity_examples.png", dpi=160, bbox_inches="tight")
plt.show()


## Notes

The most important columns in `rebalance_frequency_summary.csv` are:

- `rebalance_months`: `1` is monthly; `3` is quarterly; `12` is yearly.
- `tc_bps`: transaction cost in basis points.
- `ann_return_minus_monthly`: how much annual return changes versus monthly rebalancing at the same cost.
- `sharpe_minus_monthly`: how much Sharpe changes versus monthly rebalancing at the same cost.
- `growth_vs_monthly`: total wealth multiple divided by the monthly-rebalanced wealth multiple.
- `avg_turnover_pct`: average turnover per month after applying the chosen rebalance schedule.

If high transaction costs hurt monthly rebalancing, slower rebalancing should sometimes improve net Sharpe by lowering turnover, but it can also weaken factor exposure because the portfolio drifts away from the current labels between rebalance dates.
